# Detekcija in obravnava osamelcev — A: prikaz (telekom)
**Akademija UI za poslovne aplikacije — Modul 1, popoldanski sklop**

Ta zvezek poganja predavatelj. Vajo naredite v zvezku **`B_osamelci`** (spletna trgovina).

Cilj: razlikovati med **podatkovno napako** in **poslovno pomembnim ekstremom**.

### Ponovitev
- **Osamelec**: vrednost, ki močno odstopa od ostalih. Lahko je
  - **napaka** — vrednost je nemogoča (vnos, prenos, napačna enota), ali
  - **veljaven ekstrem** — redka, a resnična vrednost.
- **Odkrivanje:** najprej graf (histogram, škatla z brki), nato **pravilo IQR**:
  kandidati so pod Q1 − 1,5 × IQR ali nad Q3 + 1,5 × IQR (= točke zunaj brkov).
- Pravilo IQR **ne pozna poslovanja** → dodamo **domenske omejitve** (npr. naročnina ≥ 0)
  in preverimo **povezane stolpce** iste stranke ali naročila.
- **Odzivi:** popravek iz preverjenega vira, označimo kot manjkajočo (`NaN`), odstranitev (z utemeljitvijo),
  omejevanje ali logaritem, ohranitev.
- **Osamelec ni nujno napaka; včasih je ravno najzanimivejši podatek.**

### Nastavitev okolja
Zaženite spodnjo celico (izberite jo in pritisnite **Shift + Enter**). Ničesar ne spreminjajte.
Celica deluje lokalno in v Google Colabu. Koda je skrita — odprete jo s klikom na celico.

In [ ]:
#@title Nastavitev okolja (kliknite za prikaz kode)
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:           # Google Colab: prenesemo gradivo z GitHuba
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not KOREN.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotnine>=0.15", "kagglehub"], check=True)
else:                                       # lokalno: poiščemo mapo, v kateri je utils/orodja.py
    KOREN = Path.cwd()
    while not (KOREN / "utils/orodja.py").exists() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
sys.path.insert(0, str(KOREN))              # da Python najde utils/orodja.py
print("Gradivo je v:", KOREN)

### Podatki
Funkcija `poskrbi_za_podatke()` iz `utils/orodja.py` ob prvem zagonu prenese podatke s Kaggla (~1 minuta)
in jih shrani v mapo `data/`. Kako to deluje, smo pokazali v zvezku `A_prvi_stik_s_podatki`.

In [ ]:
from utils.orodja import poskrbi_za_podatke
DATA = poskrbi_za_podatke()

### Knjižnice in videz grafov
`histogram_in_skatla` smo napisali v zvezku `A_vizualizacija`; tu jo uvozimo iz `utils/orodja.py`.

In [ ]:
import numpy as np
import pandas as pd
from plotnine import *

from utils.orodja import nastavi_slog, MODRA, ORANZNA, ZELENA, SIVA, BARVE_ODHOD
from utils.orodja import histogram_in_skatla
nastavi_slog()      # enoten videz grafov (koda je v zvezku A_vizualizacija)

---
# Del A — Prikaz: telekom
## A1. Histogram in škatla z brki
Osamelec je vrednost, ki močno odstopa od ostalih. Najprej ga **vidimo** na grafu.

In [ ]:
telekom = pd.read_csv(DATA / "telekom.csv")

najvec = telekom["povp_minute"].max()
print(f"Največ: {najvec:.0f} minut na mesec")
print(f"Mesec ima 30 × 24 × 60 = {30 * 24 * 60} minut. {najvec:.0f} minut = {najvec / 30 / 60:.1f} ure klicanja na dan.")

histogram_in_skatla(telekom, "povp_minute")

Večina strank porabi nekaj sto minut na mesec, posamezne pa več tisoč — do **12 206**.
Ali je to napaka? To je ~7 ur na dan — **možno**: klicni center, podjetje, zelo zgovorna stranka.

## A2. Pravilo IQR
- **IQR** = Q3 − Q1 (razpon srednjih 50 % podatkov)
- Kandidati za osamelce: vrednosti **pod Q1 − 1,5 × IQR** ali **nad Q3 + 1,5 × IQR**
- To so ravno točke zunaj brkov na škatli z brki.

`quantile([0.25, 0.75])` vrne Q1 in Q3. `geom_vline` nariše navpično črto.

In [ ]:
q1, q3 = telekom["povp_minute"].quantile([0.25, 0.75])
iqr = q3 - q1
spodaj, zgoraj = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Q1 = {q1:.0f}, Q3 = {q3:.0f}, IQR = {iqr:.0f}")
print(f"Meji: od {spodaj:.0f} do {zgoraj:.0f} minut")

kandidati = telekom[telekom["povp_minute"] > zgoraj]
print(f"Nad zgornjo mejo: {len(kandidati)} strank ({len(kandidati) / len(telekom):.1%})")

(ggplot(telekom, aes(x="povp_minute"))
 + geom_histogram(bins=60, fill=MODRA)
 + geom_vline(xintercept=zgoraj, color=ORANZNA, linetype="dashed", size=1)
 + labs(title="povp_minute in zgornja meja IQR",
        subtitle=f"oranžna črta: Q3 + 1,5 × IQR = {zgoraj:.0f} minut", x="minute na mesec", y="število strank"))

Isti izračun bomo potrebovali za več stolpcev, zato ga zapakiramo v **funkcijo**. Funkcija **vrne** (`return`) obe meji.
Kopija je v `utils/orodja.py` — v vaji jo samo uvozimo.

In [ ]:
def meje_iqr(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

meje_iqr(telekom["povp_minute"])

Pravilo IQR je hitro, a **ne pozna poslovanja**. Poglejmo, koliko vrstic označi v različnih stolpcih:

In [ ]:
stolpci = ["povp_minute", "povp_mesecna_narocnina", "dni_z_napravo", "povp_klici_podpora", "povp_gostovanje"]
for stolpec in stolpci:
    s = telekom[stolpec].dropna()
    sp, zg = meje_iqr(s)
    oznaceni = (s < sp) | (s > zg)
    print(f"{stolpec:24s} označenih: {oznaceni.mean():6.1%}   negativnih: {(s < 0).sum()}")

Pri `povp_gostovanje` označi kar **19 %** strank. Zakaj?

In [ ]:
s = telekom["povp_gostovanje"].dropna()
print(f"Strank z 0 minutami gostovanja: {(s == 0).mean():.0%}")
print(f"Q1 = {s.quantile(0.25):.2f}, Q3 = {s.quantile(0.75):.2f}  →  zgornja meja IQR = {meje_iqr(s)[1]:.2f}")

Več kot dve tretjini strank v tujini sploh ne kliče, zato sta Q1 in Q3 skoraj 0. Že najmanjši klic iz tujine je "osamelec".

In obratno — ali pravilo IQR ujame **nemogoče** negativne naročnine? Povečajmo območje okoli 0 in uporabimo
**logaritemsko os y** (`scale_y_log10()`), da vidimo tudi stolpec z eno samo stranko. Negativne vrednosti so oranžne.

In [ ]:
s = telekom["povp_mesecna_narocnina"].dropna()
spodaj, zgoraj = meje_iqr(s)
print("Negativnih naročnin:         ", (s < 0).sum())
print("Od teh pod spodnjo mejo IQR: ", (s < spodaj).sum())

okoli_nic = pd.DataFrame({"narocnina": s[s < 5]})
(ggplot(okoli_nic, aes(x="narocnina", fill="narocnina < 0"))
 + geom_histogram(binwidth=1, boundary=0, closed="left", show_legend=False)   # stolpci [−1, 0), [0, 1) …
 + scale_fill_manual(values={True: ORANZNA, False: MODRA})
 + scale_y_log10()                                                 # viden je tudi stolpec z 1 stranko
 + geom_vline(xintercept=spodaj, linetype="dashed")
 + labs(title="povp_mesecna_narocnina: povečava okoli 0",
        subtitle=f"oranžno: negativne vrednosti, črtkano: spodnja meja IQR = {spodaj:.1f} USD",
        x="USD", y="število strank (log)"))

Spodnja meja IQR je pri −15 USD, zato pravilo ujame **samo eno** od 23 negativnih naročnin. Ostalih 22 je med mejo in 0.

**Pravilo IQR označi kandidate za pregled — ne odloči namesto nas.** Včasih označi preveč (gostovanje), včasih premalo (naročnina).

## A3. Domenske omejitve
Nekatera pravila poznamo iz poslovanja, ne iz statistike. Zapišemo jih kot pogoje in preštejemo kršitve:

In [ ]:
pravila = {
    "naročnina ne more biti negativna":  telekom["povp_mesecna_narocnina"] < 0,
    "prihodek ne more biti negativen":    telekom["povp_mesecni_prihodek"] < 0,
    "dni z napravo ne more biti < 0":     telekom["dni_z_napravo"] < 0,
    "minute ne morejo biti negativne":    telekom["povp_minute"] < 0,
}
for pravilo, krsitve in pravila.items():
    print(f"{pravilo:36s} kršitev: {krsitve.sum()}")

`dni_z_napravo` je stolpec, ki je bil v vizualizaciji najmočneje povezan z odhodom. Zato je pomembno,
kaj naredimo s 133 negativnimi vrednostmi — pri gradnji modela jih bomo označili kot manjkajoče (A6).

## A4. Preverimo povezane stolpce
Napako od pravega ekstrema pogosto ločimo tako, da pogledamo **še druge podatke iste stranke**.
`nsmallest(5, stolpec)` vrne 5 vrstic z najmanjšimi vrednostmi, `nlargest` pa z največjimi.

**Negativna naročnina** — kaj pravijo ostali stolpci?

In [ ]:
povezani = ["povp_mesecna_narocnina", "povp_mesecni_prihodek", "povp_minute", "povp_minute_6m",
            "povp_minute_prekoracitev", "mesecev_narocnine"]

telekom.nsmallest(5, "povp_mesecna_narocnina")[povezani]

- Prva stranka: naročnina **−27 USD**, prihodek pa **+49 USD**. Stolpca si nasprotujeta → verjetno napaka.
- Zadnja stranka: naročnina in prihodek sta enaka in negativna (−6,17 USD) → morda **dobropis** (vračilo denarja).

Redna mesečna naročnina ne more biti negativna. **Kaj je res, ve samo nekdo iz obračuna.**
Dokler tega ne preverimo, vrednost označimo kot manjkajočo (A6).

**Zelo visoka poraba minut** — kaj pravijo ostali stolpci?

In [ ]:
telekom.nlargest(5, "povp_minute")[povezani]

- Prihodek je pri teh strankah **visok** — stranke so res veliko klicale in za to plačale.
- Pri nekaterih je visoko tudi šestmesečno povprečje → stalno veliki uporabniki.
- Pri prvi stranki je šestmesečno povprečje veliko nižje → izjemen mesec. Redko, a možno.

Isto na grafu za vse stranke. Graf ima **dve plasti z različnimi podatki**:
siva = naključni vzorec strank, modra = vse stranke z več kot 4 000 minutami.

In [ ]:
print(f"Korelacija minute–prihodek: r = {telekom['povp_minute'].corr(telekom['povp_mesecni_prihodek']):.2f}")
vzorec = telekom.sample(4000, random_state=3)
ekstremi = telekom[telekom["povp_minute"] > 4000]

(ggplot(mapping=aes(x="povp_minute", y="povp_mesecni_prihodek"))
 + geom_point(data=vzorec, color=SIVA, alpha=0.3, size=1)
 + geom_point(data=ekstremi, color=MODRA, size=1.5)
 + labs(title="Veliko minut → visok prihodek: podatki se med seboj ujemajo",
        x="povp_minute", y="povp_mesecni_prihodek (USD)"))

## A5. Kaj izgubimo, če osamelce kar zbrišemo?

In [ ]:
_, zgoraj = meje_iqr(telekom["povp_minute"])
oznaceni = telekom["povp_minute"] > zgoraj
prihodek = telekom["povp_mesecni_prihodek"]

print(f"Označenih strank:        {oznaceni.mean():.1%}")
print(f"Njihov delež prihodka:   {prihodek[oznaceni].sum() / prihodek.sum():.1%}")
print(f"Mediana prihodka:        {prihodek[oznaceni].median():.0f} USD (ostali: {prihodek[~oznaceni].median():.0f} USD)")
print(f"Delež odhodov:           {telekom.loc[oznaceni, 'odhod'].mean():.1%} (ostali: {telekom.loc[~oznaceni, 'odhod'].mean():.1%})")

skupine = telekom.assign(skupina=np.where(oznaceni, "osamelci po IQR", "ostale stranke"))
(ggplot(skupine, aes(x="skupina", y="povp_mesecni_prihodek"))
 + geom_boxplot(outlier_shape="", fill=MODRA, alpha=0.5)
 + coord_cartesian(ylim=(0, 350))
 + labs(title="Prihodek: osamelci po minutah proti ostalim", x="", y="povp_mesecni_prihodek (USD)"))

"Osamelci" po pravilu IQR so **najbolj dragocene stranke**: 5 % strank prinese 14 % prihodka.
Če jih zbrišemo, analiza in model o njih ne vesta ničesar. **Osamelec ni nujno napaka; včasih je ravno najzanimivejši podatek.**

## A6. Možni odzivi
| Odziv | Kdaj | Primer v telekomu | Pandas |
|---|---|---|---|
| Popravek iz preverjenega vira | Imamo dostop do izvornega sistema | Naročnino preverimo v obračunu | `df.loc[pogoj, stolpec] = nova_vrednost` |
| Označimo kot manjkajočo (NaN) | Vrednost je nemogoča, prave ne poznamo | Negativna naročnina, negativni dnevi | `s.mask(pogoj)` |
| Odstranitev | Utemeljeno: vrstica ne sodi v analizo | Testni računi, podjetja v analizi zasebnih strank | `df[~pogoj]` |
| Omejevanje ali transformacija | Veljavna vrednost, a model je občutljiv na ekstreme | Minute: omejimo na 99. percentil ali log | `s.clip(upper=meja)`, `np.log1p(s)` |
| Ohranitev | Veljavna vrednost | Minute pri analizi prihodka | — |

**Nemogoče vrednosti → manjkajoče.** `mask(pogoj)` zamenja z `NaN` vse vrednosti, kjer je pogoj `True`.
Potem jih obravnavamo kot v prejšnjem bloku.

In [ ]:
narocnina = telekom["povp_mesecna_narocnina"]
telekom["povp_mesecna_narocnina"] = narocnina.mask(narocnina < 0)

print("Negativnih po popravku:", (telekom["povp_mesecna_narocnina"] < 0).sum())
print("Manjkajočih:", narocnina.isna().sum(), "→", telekom["povp_mesecna_narocnina"].isna().sum())

**Veljavni ekstremi → omejimo ali transformiramo** (samo, če to potrebuje model).
- Omejevanje (*capping*): `clip(upper=meja)` vse nad mejo postavi na mejo, tu na 99. percentil.
- Logaritem: `np.log1p(x)` = log(1 + x) ohrani vrstni red strank, a "stisne" dolg rep (`1 +`, ker log(0) ne obstaja).

Vse tri različice zložimo v eno tabelo in jih narišemo kot **podgrafe** (`facet_wrap`), vsak s svojo osjo (`scales="free"`):

In [ ]:
minute = telekom["povp_minute"].dropna()
p99 = minute.quantile(0.99)
print(f"Omejenih vrednosti (nad 99. percentilom = {p99:.0f} minut): {(minute > p99).sum()}")

razlicice = pd.concat([
    pd.DataFrame({"minute": minute, "razlicica": "1) izvirno"}),
    pd.DataFrame({"minute": minute.clip(upper=p99), "razlicica": f"2) clip(upper={p99:.0f})"}),
    pd.DataFrame({"minute": np.log1p(minute), "razlicica": "3) np.log1p"}),
])
(ggplot(razlicice, aes(x="minute"))
 + geom_histogram(bins=50, fill=MODRA)
 + facet_wrap("~razlicica", scales="free")
 + labs(title="Minute: izvirno, omejeno in logaritmirano", x="", y="število strank")
 + theme(figure_size=(11, 3.5)))

> Meje za omejevanje (npr. 99. percentil) so parametri predprocesiranja — tako kot mediana v prejšnjem bloku
> jih pri gradnji modela izračunamo **samo na učnih podatkih**.

---
Naprej: zvezek **`B_osamelci`** — osamelci v spletni trgovini.